In [7]:
# ==========================================
# AUGMENTASI BALANCED (INPUT: dataset_merge)
# OUTPUT: dataset_aug1
# - Ambil kelas otomatis dari folder
# - Hanya augment kelas yang kurang
# ==========================================

import os
import shutil
import random
from tqdm import tqdm
from PIL import Image, ImageEnhance


In [ ]:

# Path
INPUT_DIR = "../dataset_merged"
OUTPUT_DIR = "../dataset_merged_aug"

# ==========================================
# AMBIL KELAS OTOMATIS
# ==========================================

def get_classes(dir_path):
    return sorted([
        d for d in os.listdir(dir_path)
        if os.path.isdir(os.path.join(dir_path, d))
    ])

CLASSES = get_classes(INPUT_DIR)
print("Kelas:", CLASSES)

# ==========================================
# COPY DATA AWAL KE OUTPUT
# ==========================================

for cls in CLASSES:
    src = os.path.join(INPUT_DIR, cls)
    dst = os.path.join(OUTPUT_DIR, cls)
    os.makedirs(dst, exist_ok=True)

    files = [f for f in os.listdir(src) if f.lower().endswith((".jpg", ".png", ".jpeg"))]

    for f in tqdm(files, desc=f"Copy {cls}"):
        shutil.copy2(os.path.join(src, f), os.path.join(dst, f))

# ==========================================
# HITUNG JUMLAH
# ==========================================

class_counts = {}
for cls in CLASSES:
    path = os.path.join(OUTPUT_DIR, cls)
    class_counts[cls] = len(os.listdir(path))

max_count = max(class_counts.values())

print("\nJumlah sebelum augmentasi:")
for k,v in class_counts.items():
    print(f"{k}: {v}")
print("Target:", max_count)


Kelas: ['alur', 'lubang', 'retak', 'tidak_rusak']


Copy tidak_rusak: 100%|██████████| 70/70 [00:00<00:00, 119.71it/s]


Jumlah sebelum augmentasi:
alur: 95
lubang: 621
retak: 243
tidak_rusak: 70
Target: 621


In [ ]:

# ==========================================
# AUGMENTASI
# ==========================================

AUG_TYPES = ["flip", "flip_v", "bright", "dark", "zoom", "rotate"]

for cls in CLASSES:
    cls_path = os.path.join(OUTPUT_DIR, cls)
    images = [f for f in os.listdir(cls_path) if f.lower().endswith((".jpg", ".png", ".jpeg"))]

    total_now = len(images)
    total_needed = max_count - total_now

    if total_needed <= 0:
        print(f"{cls} sudah cukup ({total_now})")
        continue

    print(f"\nAugmenting {cls}, butuh: {total_needed}")

    total_generated = 0

    while total_generated < total_needed:
        file = random.choice(images)
        img_path = os.path.join(cls_path, file)

        img_pil = Image.open(img_path).convert("RGB")
        base_name, ext = os.path.splitext(file)

        aug_type = random.choice(AUG_TYPES)
        aug_img = img_pil.copy()

        if aug_type == "flip":
            aug_img = aug_img.transpose(Image.FLIP_LEFT_RIGHT)

        elif aug_type == "flip_v":
            aug_img = aug_img.transpose(Image.FLIP_TOP_BOTTOM)

        elif aug_type == "bright":
            aug_img = ImageEnhance.Brightness(aug_img).enhance(1.7)

        elif aug_type == "dark":
            aug_img = ImageEnhance.Brightness(aug_img).enhance(0.3)

        elif aug_type == "zoom":
            zoom_factor = random.uniform(1.2, 1.6)
            w, h = aug_img.size
            new_w, new_h = int(w / zoom_factor), int(h / zoom_factor)
            left = (w - new_w) // 2
            top = (h - new_h) // 2
            aug_img = aug_img.crop((left, top, left + new_w, top + new_h)).resize((w, h))
            
        elif aug_type == "rotate":
            aug_img = aug_img.rotate(random.randint(-20, 20), fillcolor=(128,128,128))

        new_name = f"{base_name}_aug_{aug_type}_{total_generated+1:03d}{ext.lower()}"
        aug_img.save(os.path.join(cls_path, new_name))

        total_generated += 1

print("\nSelesai!")

# ==========================================
# CEK AKHIR
# ==========================================

print("\nJumlah akhir:")
for cls in CLASSES:
    total = len(os.listdir(os.path.join(OUTPUT_DIR, cls)))
    print(f"{cls}: {total}")



Augmenting alur, butuh: 526
lubang sudah cukup (621)

Augmenting retak, butuh: 378

Augmenting tidak_rusak, butuh: 551

Selesai!

Jumlah akhir:
alur: 621
lubang: 621
retak: 621
tidak_rusak: 621
